# TCN Final Evaluation Notebook

This notebook is evaluation-only and designed for final model selection, ablations, and benchmarking.
All runtime variables are isolated with the `eval_` prefix to avoid conflicts with training notebooks.

## 1) Connect to Colab VM and sync repository
Run this first in a fresh Colab runtime.

In [1]:
import os

EVAL_REPO_URL = "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git"
EVAL_REPO_DIR = "/content/tcn_tape_vectorized_version_clean"

if not os.path.exists(f"{EVAL_REPO_DIR}/.git"):
    !git clone {EVAL_REPO_URL} {EVAL_REPO_DIR}

%cd /content/tcn_tape_vectorized_version_clean
!git fetch origin
!git reset --hard origin/main

/content/tcn_tape_vectorized_version_clean
HEAD is now at 398e677 Adjust position cap to 20% and update notebook checkpoint threshold


## 2) Optional: mount Drive and restore saved results zip
Set `EVAL_RESTORE_FROM_ZIP=True` only when needed.

In [2]:
from pathlib import Path

EVAL_RESTORE_FROM_ZIP = True
EVAL_ZIP_PATH = "/content/drive/MyDrive/tcn_tape_vectorized_run1.zip"

if EVAL_RESTORE_FROM_ZIP:
    from google.colab import drive
    drive.mount('/content/drive')

    zip_path = Path(EVAL_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip not found: {zip_path}")

    !mkdir -p /content/tcn_tape_vectorized_version_clean
    !unzip -q -o {zip_path} -d /content/tcn_tape_vectorized_version_clean
    print("✅ Restored results from zip")
else:
    print("ℹ️ EVAL_RESTORE_FROM_ZIP=False")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Restored results from zip


In [3]:
from pathlib import Path

EVAL_ZIP_PATH = Path("/content/drive/MyDrive/tcn_tape_vectorized_run1.zip")

# 1) Verify zip exists
print("zip exists:", EVAL_ZIP_PATH.exists())
if not EVAL_ZIP_PATH.exists():
    raise FileNotFoundError(EVAL_ZIP_PATH)

# 2) Inspect zip top-level structure
!unzip -l "{EVAL_ZIP_PATH}" | head -n 40

# 3) Extract to /content (clean target)
!mkdir -p /content/eval_restore
!unzip -q -o "{EVAL_ZIP_PATH}" -d /content/eval_restore

# 4) Auto-detect correct results root (aligned with training saves)
candidates = [
    Path("/content/eval_restore/tcn_fusion_results"),
    Path("/content/eval_restore/tcn_tape_vectorized_version_clean/tcn_fusion_results"),
    Path("/content/eval_restore/tcn_tape_vectorized_version_clean"),  # project root fallback
]
resolved_candidates = []
for c in candidates:
    root = c if c.name == "tcn_fusion_results" else (c / "tcn_fusion_results")
    if root not in resolved_candidates:
        resolved_candidates.append(root)

for c in resolved_candidates:
    has_logs = (c / "logs").exists()
    has_hw = (c / "high_watermark_checkpoints").exists()
    n_actor = len(list((c / "high_watermark_checkpoints").glob("*_actor.weights.h5"))) if has_hw else len(list(c.rglob("*_actor.weights.h5")))
    print(c, "logs:", has_logs, "hw:", has_hw, "actors:", n_actor)

EVAL_RESULTS_ROOT = next(
    c for c in resolved_candidates
    if (c / "logs").exists()
    and (c / "high_watermark_checkpoints").exists()
    and len(list((c / "high_watermark_checkpoints").glob("*_actor.weights.h5"))) > 0
)
print("✅ EVAL_RESULTS_ROOT =", EVAL_RESULTS_ROOT)


zip exists: True
Archive:  /content/drive/MyDrive/tcn_tape_vectorized_run1.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-02-24 14:47   tcn_fusion_results/
        0  2026-02-24 18:54   tcn_fusion_results/logs/
 33264205  2026-02-24 18:54   tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260224_144630_step_diagnostics.csv
    71456  2026-02-24 18:54   tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260224_144630_metadata.json
    12286  2026-02-24 18:54   tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260224_144630_summary.csv
    12377  2026-02-24 18:54   tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260224_144630_episodes.csv
     9609  2026-02-24 14:46   tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260224_144630_active_feature_manifest.json
        0  2026-02-24 18:52   tcn_fusion_results/high_watermark_checkpoints/
  4237192  2026-02-24 14:51   tcn_f

In [4]:
# Install project requirements in Colab VM
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/tcn_tape_vectorized_version_clean")
REQ_FILE = REPO_DIR / "requirements.txt"

if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

print("Using python:", sys.executable)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)], check=True)

print("✅ Requirements installed")

Using python: /usr/bin/python3
✅ Requirements installed


In [4]:
import tensorflow as tf
tf.keras.mixed_precision.set_global_policy("float32")
print(tf.keras.mixed_precision.global_policy())

<DTypePolicy "float32">


## 3) Imports

In [5]:
import copy
import json
import re
from dataclasses import replace
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from src.config import get_active_config
from src.data_utils import DataProcessor
from src.notebook_helpers.tcn_phase1 import (
    prepare_phase1_dataset,
    create_experiment6_result_stub,
    evaluate_experiment6_checkpoint,
    load_training_metadata_into_config,
    build_evaluation_track_summary,
    build_ablation_table,
    compare_agent_vs_baseline,
    Phase1Dataset,
    split_dataset_by_date,
    identify_covariance_columns,
)

## 4) Evaluation run settings
Adjust once here.

In [6]:
EVAL_RANDOM_SEED = 42
# Keep auto-detected root from extraction cell unless explicitly overridden.
EVAL_RESULTS_ROOT = Path(globals().get("EVAL_RESULTS_ROOT", "/content/eval_restore/tcn_fusion_results"))

# Deterministic policy mode: 'mean' is recommended for stable ranking.
EVAL_DETERMINISTIC_MODE = 'mean'
EVAL_STOCHASTIC_MODE = "sample"

# Stochastic robustness checks per checkpoint.
EVAL_NUM_STOCHASTIC_RUNS = 50
EVAL_STOCHASTIC_EPISODE_LIMIT = None

# Selection for ablation basket
EVAL_TOP_HW = 8         # high-watermark checkpoints by filename Sharpe tag
EVAL_TOP_PERIODIC = 4   # periodic step checkpoints by most recent step
EVAL_INCLUDE_ROOT = True
EVAL_INCLUDE_RARE = False

# Save outputs
EVAL_SAVE_LOGS = True
EVAL_SAVE_ARTIFACTS = True

In [7]:
print((EVAL_RESULTS_ROOT / "logs").exists())
print(len(list(EVAL_RESULTS_ROOT.rglob("*_actor.weights.h5"))))

True
83


In [8]:
print("root exists:", EVAL_RESULTS_ROOT.exists())
print("logs exists:", (EVAL_RESULTS_ROOT / "logs").exists())
print("actor ckpts:", len(list(EVAL_RESULTS_ROOT.rglob("*_actor.weights.h5"))))

root exists: True
logs exists: True
actor ckpts: 83


## 5) Build evaluation dataset and load latest metadata config

In [9]:
if "eval_phase1_data" in globals():
    del eval_phase1_data

In [10]:
# ============================================================================
# EVAL CONFIG + FEATURE LOCK (metadata-trained layout) + DATASET BUILD
# ============================================================================

from src.config import get_active_config
from src.data_utils import DataProcessor
from src.notebook_helpers.tcn_phase1 import (
    load_training_metadata_into_config,
    Phase1Dataset,
    prepare_phase1_dataset,
    split_dataset_by_date,
    identify_covariance_columns,
)

if not EVAL_RESULTS_ROOT.exists():
    raise FileNotFoundError(f"Missing results root: {EVAL_RESULTS_ROOT}")


def eval_extract_trained_state_layout(metadata_dict: dict):
    arch = metadata_dict.get("Architecture_Settings", {}) or {}

    # Try effective first, then template
    effective = arch.get("agent_params_effective", {}) or {}
    template = arch.get("agent_params_template", {}) or {}

    layout = effective.get("state_layout")
    if not isinstance(layout, dict) or not layout:
        layout = template.get("state_layout")

    if not isinstance(layout, dict) or not layout:
        raise ValueError("Could not find state_layout in metadata (agent_params_effective/template).")

    active_cols = layout.get("active_feature_columns")
    if not isinstance(active_cols, list) or not active_cols:
        raise ValueError("state_layout.active_feature_columns missing/empty in metadata.")

    return layout, list(dict.fromkeys(active_cols))


def eval_apply_metadata_feature_lock(cfg, trained_active_feature_columns):
    """
    Lock feature selection to the trained layout while preserving project-level drops.

    Important: include trained-only runtime groups (e.g., Actuarial_*) in the
    candidate universe so counts and lock math stay aligned with training.
    """
    probe_cfg = copy.deepcopy(cfg)
    probe_fp = probe_cfg.setdefault("feature_params", {})
    probe_fs = probe_fp.setdefault("feature_selection", {})
    probe_fs["disable_features"] = False
    probe_fs["disabled_features"] = []

    probe = DataProcessor(probe_cfg)
    core_all_cols = list(dict.fromkeys(probe.get_feature_columns("phase1")))

    # Ensure runtime-only trained columns are represented in lock universe.
    for col in trained_active_feature_columns:
        if col not in core_all_cols:
            core_all_cols.append(col)

    trained_set = set(trained_active_feature_columns)
    from_core_gap = {c for c in core_all_cols if c not in trained_set}

    # Preserve preconfigured drops (audit + curated disables).
    existing_disabled = set(
        cfg.get("feature_params", {})
        .get("feature_selection", {})
        .get("disabled_features", [])
    )
    disabled = sorted(existing_disabled.union(from_core_gap))

    fp = cfg.setdefault("feature_params", {})
    fs = fp.setdefault("feature_selection", {})
    fs["disable_features"] = True
    fs["disabled_features"] = disabled

    return core_all_cols, disabled


def eval_bind_trained_feature_layout(processor, trained_active_feature_columns):
    """
    Force eval-time feature list to match the exact trained state layout.
    This prevents runtime-family omissions (e.g., actuarial) when loading
    from a pre-normalized CSV without re-running full feature engineering.
    """
    trained_cols = list(dict.fromkeys(trained_active_feature_columns))
    base_get_feature_columns = processor.get_feature_columns

    def _locked_get_feature_columns(phase='phase1'):
        if str(phase).lower() == 'phase1':
            return list(trained_cols)
        return base_get_feature_columns(phase)

    processor.get_feature_columns = _locked_get_feature_columns
    return processor


# ------------------------------------------------------------------
# Build eval config from latest metadata
# ------------------------------------------------------------------
eval_config = copy.deepcopy(get_active_config("phase1"))

eval_logs_dir = EVAL_RESULTS_ROOT / "logs"
meta_files = sorted(eval_logs_dir.glob("*_metadata.json"), key=lambda p: p.stat().st_mtime, reverse=True)
if not meta_files:
    raise FileNotFoundError(f"No metadata JSON in {eval_logs_dir}")

EVAL_METADATA_PATH = meta_files[0]
print("📄 Using metadata:", EVAL_METADATA_PATH)

with open(EVAL_METADATA_PATH, "r", encoding="utf-8") as f:
    eval_metadata = json.load(f)

eval_config = load_training_metadata_into_config(
    EVAL_METADATA_PATH,
    copy.deepcopy(eval_config),
    verbose=True,
)

# Hard requirement for aligned evaluation: no fundamentals, actuarial ON.
eval_fp = eval_config.setdefault("feature_params", {})
eval_fund_cfg = eval_fp.setdefault("fundamental_features", {})
eval_act_cfg = eval_fp.setdefault("actuarial_params", {})
eval_fund_cfg["enabled"] = False
eval_act_cfg["enabled"] = True
print("   eval fundamental_features.enabled:", bool(eval_fund_cfg.get("enabled", False)))
print("   eval actuarial_params.enabled:", bool(eval_act_cfg.get("enabled", False)))

trained_num_parallel_envs = (
    (eval_metadata.get("Training_Hyperparameters", {}) or {}).get("num_parallel_envs")
    or (eval_metadata.get("Run_Context", {}) or {}).get("num_parallel_envs")
)
eval_config.setdefault("training_params", {})
eval_config["training_params"]["num_parallel_envs"] = 1  # evaluation remains single-env
print("   trained num_parallel_envs:", trained_num_parallel_envs)
print("   eval num_parallel_envs (forced):", eval_config["training_params"]["num_parallel_envs"])

# Enforce architecture family used by checkpoints
eval_config["agent_params"]["actor_critic_type"] = "TCN_FUSION"
eval_config["agent_params"]["use_fusion"] = True
eval_config["agent_params"]["use_attention"] = False

# A2/A3/A4 compatibility defaults (legacy runs remain loadable)
eval_config["agent_params"].setdefault("fusion_cross_asset_mixer_enabled", False)
eval_config["agent_params"].setdefault("fusion_cross_asset_mixer_layers", 1)
eval_config["agent_params"].setdefault("fusion_cross_asset_mixer_expansion", 2.0)
eval_config["agent_params"].setdefault("fusion_cross_asset_mixer_dropout", 0.1)
eval_config["agent_params"].setdefault("fusion_alpha_head_hidden_dims", [])
eval_config["agent_params"].setdefault("fusion_alpha_head_dropout", 0.1)

print("   Fusion mixer cfg:", {k: eval_config["agent_params"][k] for k in [
    "fusion_cross_asset_mixer_enabled",
    "fusion_cross_asset_mixer_layers",
    "fusion_cross_asset_mixer_expansion",
    "fusion_cross_asset_mixer_dropout",
]})
print("   Fusion alpha head cfg:", eval_config["agent_params"]["fusion_alpha_head_hidden_dims"],
      "| dropout:", eval_config["agent_params"]["fusion_alpha_head_dropout"])


# New-architecture compatibility defaults (safe for older metadata too)
arch_effective = ((eval_metadata.get("Architecture_Settings", {}) or {}).get("agent_params_effective", {}) or {})
for _k, _fallback in {
    "recurrent_memory_enabled": False,
    "recurrent_memory_units": 64,
    "recurrent_memory_dropout": 0.1,
    "regime_conditioning_enabled": False,
    "regime_conditioning_hidden_dim": 32,
    "regime_conditioning_dropout": 0.0,
    "state_augmentation_enabled": False,
    "distributional_critic_enabled": False,
    "distributional_num_quantiles": 17,
}.items():
    if _k in arch_effective:
        eval_config["agent_params"][_k] = arch_effective[_k]
    else:
        eval_config["agent_params"].setdefault(_k, _fallback)

eval_ppo = eval_config["agent_params"].setdefault("ppo_params", {})
for _k, _fallback in {
    "popart_enabled": False,
    "popart_min_std": 1e-3,
    "multi_horizon_reward_enabled": False,
    "multi_horizon_reward_coef": 0.0,
    "multi_horizon_reward_horizons": [21, 63, 126, 252],
    "multi_horizon_reward_weights": [0.25, 0.25, 0.25, 0.25],
}.items():
    eval_ppo.setdefault(_k, _fallback)

print("   Memory/regime/distributional cfg:", {
    "recurrent_memory_enabled": eval_config["agent_params"].get("recurrent_memory_enabled"),
    "regime_conditioning_enabled": eval_config["agent_params"].get("regime_conditioning_enabled"),
    "state_augmentation_enabled": eval_config["agent_params"].get("state_augmentation_enabled"),
    "distributional_critic_enabled": eval_config["agent_params"].get("distributional_critic_enabled"),
    "distributional_num_quantiles": eval_config["agent_params"].get("distributional_num_quantiles"),
})
print("   PPO PopArt/reward-decomp cfg:", {
    "popart_enabled": eval_ppo.get("popart_enabled"),
    "multi_horizon_reward_enabled": eval_ppo.get("multi_horizon_reward_enabled"),
    "multi_horizon_reward_coef": eval_ppo.get("multi_horizon_reward_coef"),
})


# Extract trained state layout and lock features to it
trained_state_layout, trained_active_feature_columns = eval_extract_trained_state_layout(eval_metadata)

# Enforce no-fundamentals policy for aligned training/evaluation.
trained_fund_cols = [c for c in trained_active_feature_columns if str(c).startswith("Fundamental_")]
if trained_fund_cols:
    raise ValueError(
        "Checkpoint metadata still includes Fundamental_ columns. "
        "Use checkpoints trained after fundamental removal. "
        f"Sample: {trained_fund_cols[:8]}"
    )
required_act_cols = [
    "Actuarial_Expected_Recovery",
    "Actuarial_Prob_30d",
    "Actuarial_Prob_60d",
    "Actuarial_Reserve_Severity",
]
missing_required_act = [c for c in required_act_cols if c not in trained_active_feature_columns]
if missing_required_act:
    raise ValueError(
        "Checkpoint metadata missing required actuarial columns: "
        f"{missing_required_act}"
    )

# Keep layout in config for agent reconstruction compatibility
eval_config["agent_params"]["state_layout"] = copy.deepcopy(trained_state_layout)
eval_config["agent_params"]["asset_feature_dim"] = int(trained_state_layout.get("asset_feature_dim", 0) or 0)
eval_config["agent_params"]["global_feature_dim"] = int(trained_state_layout.get("global_feature_dim", 0) or 0)
eval_config["agent_params"]["num_assets"] = int(trained_state_layout.get("num_assets", 10) or 10)

core_all_cols, eval_disabled_features = eval_apply_metadata_feature_lock(
    eval_config, trained_active_feature_columns
)

act_trained = [c for c in trained_active_feature_columns if c.startswith("Actuarial_")]
configured_allowlist = list(dict.fromkeys(
    eval_config.get("feature_params", {}).get("feature_selection", {}).get("active_features_allowlist", []) or []
))
if configured_allowlist:
    overlap = len(set(trained_active_feature_columns) & set(configured_allowlist))
    print("   configured audit allowlist:", len(configured_allowlist))
    print("   overlap(trained vs allowlist):", overlap)
    if overlap != len(trained_active_feature_columns):
        print("   ⚠️ Metadata-trained layout differs from configured allowlist (expected for legacy runs).")
print("✅ Eval metadata feature lock applied")
print("   trained active_feature_columns:", len(trained_active_feature_columns))
print("   trained actuarial columns:", len(act_trained), act_trained)
print("   core feature_columns (+runtime groups):", len(core_all_cols))
print("   disabled_features:", len(eval_disabled_features))
print("   expected active after lock:", len(core_all_cols) - len(eval_disabled_features))
print("   state_layout asset/global dims:",
      trained_state_layout.get("asset_feature_dim"),
      trained_state_layout.get("global_feature_dim"))

# ------------------------------------------------------------------
# Build eval dataset from SAVED normalized master features (no rebuild)
# ------------------------------------------------------------------
EVAL_USE_SAVED_NORMALIZED = True
EVAL_FORCE_REBUILD_PHASE1 = True  # IMPORTANT: avoid stale globals from prior runs

if EVAL_FORCE_REBUILD_PHASE1 and "eval_phase1_data" in globals():
    del eval_phase1_data
    print("🧹 Cleared stale eval_phase1_data from runtime")

if "eval_phase1_data" in globals():
    print("ℹ️ Reusing eval_phase1_data from current runtime")
else:
    if not EVAL_USE_SAVED_NORMALIZED:
        eval_phase1_data = prepare_phase1_dataset(eval_config, force_download=False)
        eval_phase1_data.data_processor = eval_bind_trained_feature_layout(
            eval_phase1_data.data_processor,
            trained_active_feature_columns,
        )
    else:
        normalized_candidates = [
            EVAL_RESULTS_ROOT / "data" / "master_features_NORMALIZED.csv",
            Path("/content/tcn_tape_vectorized_version_clean/tcn_tape_vectorized_run1/data/master_features_NORMALIZED.csv"),
            Path(eval_config.get("BASE_DATA_PATH", "/content/tcn_tape_vectorized_version_clean/data")) / "master_features_NORMALIZED.csv",
            Path("/content/tcn_tape_vectorized_version_clean/data/master_features_NORMALIZED.csv"),
        ]
        normalized_path = next((p for p in normalized_candidates if p.exists()), None)
        if normalized_path is None:
            raise FileNotFoundError(
                "Could not find master_features_NORMALIZED.csv in expected locations:\n"
                + "\n".join(str(p) for p in normalized_candidates)
            )

        print("📦 Loading normalized master from:", normalized_path)
        master_df_norm = pd.read_csv(normalized_path)

        if "Date" not in master_df_norm.columns:
            raise ValueError("Normalized CSV missing required 'Date' column")
        if "Ticker" not in master_df_norm.columns:
            raise ValueError("Normalized CSV missing required 'Ticker' column")

        master_df_norm["Date"] = pd.to_datetime(
            master_df_norm["Date"], utc=True, errors="coerce"
        ).dt.tz_localize(None)
        master_df_norm = master_df_norm.dropna(subset=["Date"]).sort_values(["Date", "Ticker"]).reset_index(drop=True)

        analysis_start = pd.to_datetime(eval_config.get("ANALYSIS_START_DATE", "2003-09-02"))
        analysis_end = pd.to_datetime(eval_config.get("ANALYSIS_END_DATE", "2025-09-01"))
        master_df_norm = master_df_norm[
            (master_df_norm["Date"] >= analysis_start) &
            (master_df_norm["Date"] <= analysis_end)
        ].copy()

        missing_trained = [c for c in trained_active_feature_columns if c not in master_df_norm.columns]
        if missing_trained:
            raise ValueError(
                f"Saved normalized CSV missing {len(missing_trained)} trained active columns. "
                f"Sample: {missing_trained[:10]}"
            )

        eval_processor = DataProcessor(eval_config)
        eval_processor = eval_bind_trained_feature_layout(eval_processor, trained_active_feature_columns)

        split_date = eval_config.get("TRAIN_TEST_SPLIT_DATE")
        if split_date:
            train_df, test_df, train_end_date, test_start_date = split_dataset_by_date(
                master_df_norm, date_column="Date", split_date=split_date
            )
        else:
            train_df, test_df, train_end_date, test_start_date = split_dataset_by_date(
                master_df_norm, date_column="Date", train_fraction=0.8
            )

        eval_phase1_data = Phase1Dataset(
            master_df=master_df_norm,
            train_df=train_df,
            test_df=test_df,
            scalers={},
            train_end_date=train_end_date,
            test_start_date=test_start_date,
            covariance_columns=identify_covariance_columns(master_df_norm.columns),
            data_processor=eval_processor,
        )

        print("✅ Built eval_phase1_data from saved normalized master")
        print("   Train shape:", eval_phase1_data.train_df.shape)
        print("   Test shape:", eval_phase1_data.test_df.shape)
        print("   Covariance cols:", len(eval_phase1_data.covariance_columns))

# Final alignment checks (must pass for both training/eval parity)
_eval_used = list(dict.fromkeys(eval_phase1_data.data_processor.get_feature_columns("phase1")))
_eval_act_cols = [c for c in _eval_used if str(c).startswith("Actuarial_")]
_eval_fund_cols = [c for c in _eval_used if str(c).startswith("Fundamental_")]
if _eval_fund_cols:
    raise RuntimeError(f"Evaluation feature list still contains fundamentals: {_eval_fund_cols[:10]}")
if not _eval_act_cols:
    raise RuntimeError("Evaluation feature list missing actuarial columns")

_eval_master = eval_phase1_data.master_df
_missing_eval_act = [c for c in _eval_act_cols if c not in _eval_master.columns]
if _missing_eval_act:
    raise RuntimeError(f"Actuarial columns missing in eval master_df: {_missing_eval_act}")
_eval_act_non_null = {c: int(_eval_master[c].notna().sum()) for c in _eval_act_cols}
if any(v == 0 for v in _eval_act_non_null.values()):
    raise RuntimeError(f"Actuarial columns present but empty in eval master_df: {_eval_act_non_null}")

print("✅ Eval alignment checks passed")
print("   actuarial columns:", _eval_act_cols)
print("   actuarial non-null:", _eval_act_non_null)
print("   fundamental columns in eval feature list:", len(_eval_fund_cols))



📄 Using metadata: /content/eval_restore/tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260224_144630_metadata.json
✅ Applied training metadata to config
   Metadata: /content/eval_restore/tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260224_144630_metadata.json
   Run timestamp: 20260224_144630
   Architecture: TCN_FUSION
   TCN stack: filters=[64, 96, 128, 128, 128] | kernel=5 | dilations=[1, 2, 4, 8, 16] | dropout=0.15
   Fusion core: embed=128 | heads=4 | dropout=0.1
   Fusion mixer (A4): enabled=True | layers=2 | expansion=2.0 | dropout=0.1
   Fusion alpha head (A3): dims=[128, 64] | dropout=0.05
   Dirichlet controls: activation=softplus | temperature=1.0 | adaptive_temp=True | adaptive_base=0.9 | adaptive_slope=0.6 | adaptive_range=[0.8, 2.5] | alpha_cap=20.0 | epsilon={'max': 0.2, 'min': 0.02}
   Turnover target: 0.35
   DSR scalar: 2.0
   PPO update timesteps: scheduled
   Episode length curriculum: True
   RA-KL enabled: False
   Determinis

In [11]:
print(eval_phase1_data.train_df.shape, eval_phase1_data.test_df.shape)
print("Date min/max test:", eval_phase1_data.test_df["Date"].min(), eval_phase1_data.test_df["Date"].max())

_eval_used = list(dict.fromkeys(eval_phase1_data.data_processor.get_feature_columns("phase1")))
_eval_act = [c for c in _eval_used if c.startswith("Actuarial_")]
print("Eval phase1 feature count:", len(_eval_used))
print("Eval actuarial features:", len(_eval_act), _eval_act)


(43867, 103) (11030, 103)
Date min/max test: 2021-04-12 00:00:00 2025-08-29 00:00:00
Eval phase1 feature count: 49
Eval actuarial features: 4 ['Actuarial_Expected_Recovery', 'Actuarial_Prob_30d', 'Actuarial_Prob_60d', 'Actuarial_Reserve_Severity']


## 6) Inspect latest training CSV logs (for diagnostics context)

In [12]:
if not eval_logs_dir.exists():
    raise FileNotFoundError(f"Missing logs dir: {eval_logs_dir}")

def eval_latest_csv(pattern):
    files = sorted(eval_logs_dir.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None

eval_latest_episodes_csv = eval_latest_csv('*episodes*.csv')
eval_latest_step_diag_csv = eval_latest_csv('*step_diagnostics*.csv')
eval_latest_summary_csv = eval_latest_csv('*summary*.csv')

print('episodes:', eval_latest_episodes_csv)
print('step diagnostics:', eval_latest_step_diag_csv)
print('summary:', eval_latest_summary_csv)

if eval_latest_episodes_csv:
    eval_episodes_df = pd.read_csv(eval_latest_episodes_csv)
    display(eval_episodes_df.tail(5))

episodes: /content/eval_restore/tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260224_144630_episodes.csv
step diagnostics: /content/eval_restore/tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260224_144630_step_diagnostics.csv
summary: /content/eval_restore/tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260224_144630_summary.csv


,update,timestep,episode,elapsed_time,episode_return_pct,episode_sharpe,episode_sortino,episode_max_dd,episode_volatility,episode_win_rate,...,actor_grad_norm,critic_grad_norm,alpha_min,alpha_max,alpha_mean,ratio_mean,ratio_std,drawdown_lambda_peak,episode_length,termination_reason
4,50,50400,100,7086.817238,10.788780,0.184727,0.240692,41.774384,0.260453,54.172185,...,0.699065,4.628670,0.701674,1.270630,0.999802,0.996185,0.266473,1.804926,756.0,episode_limit
5,60,65520,120,9235.742078,45.819100,0.813336,1.099653,14.471749,0.143066,52.980132,...,0.771408,8.484270,0.846683,1.190700,1.047157,0.998322,0.176689,0.000000,756.0,episode_limit
6,70,80640,140,11767.893448,42.502003,0.687820,0.932812,15.293590,0.162261,54.569536,...,0.844751,2.149437,0.695684,1.193269,1.054994,1.004680,0.197667,0.000000,756.0,episode_limit
7,80,95760,160,14482.878729,17.310757,0.209846,0.265803,30.003877,0.149354,55.213505,...,0.948086,0.945556,0.589375,1.219248,1.008693,0.995991,0.245442,0.839733,1008.0,episode_limit
8,83,100000,164,14899.411969,57.606483,0.812224,1.197594,17.241432,0.125487,55.908640,...,0.829589,1.660968,0.570634,1.205132,1.040749,0.995921,0.237400,0.153413,1008.0,episode_limit


## 7) Checkpoint discovery and ablation basket construction

In [13]:
# ============================================================================
# CHECKPOINT DISCOVERY + ABLATION BASKET (with forced episode includes)
# ============================================================================

#import re
#from pathlib import Path
#import numpy as np
#import pandas as pd

# -----------------------
# Config knobs
# -----------------------
EVAL_TOP_ROOT = globals().get("EVAL_TOP_ROOT", 2)
EVAL_TOP_HW = globals().get("EVAL_TOP_HW", 8)
EVAL_TOP_PERIODIC = globals().get("EVAL_TOP_PERIODIC", 4)
EVAL_TOP_RARE = globals().get("EVAL_TOP_RARE", 3)

EVAL_INCLUDE_ROOT = globals().get("EVAL_INCLUDE_ROOT", True)
EVAL_INCLUDE_RARE = globals().get("EVAL_INCLUDE_RARE", False)

# Force-include these episodes even if they are not in top-sharpe basket
# Hard-set forced episodes for this run
EVAL_FORCE_EPISODES = [5, 25, 50, 45, 40, 154, 164]
EVAL_FORCE_EPISODES = sorted({int(x) for x in EVAL_FORCE_EPISODES})

if "EVAL_RESULTS_ROOT" not in globals():
    raise NameError("EVAL_RESULTS_ROOT is not defined")

EVAL_RESULTS_ROOT = Path(EVAL_RESULTS_ROOT)
if not EVAL_RESULTS_ROOT.exists():
    raise FileNotFoundError(f"Missing results root: {EVAL_RESULTS_ROOT}")


# -----------------------
# Parse helpers
# -----------------------
def _ckpt_name(x) -> str:
    return x.name if isinstance(x, Path) else str(x)


def eval_parse_sharpe_from_name(x):
    # Supports: ..._shp1p234... and ..._shm0p456...
    name = _ckpt_name(x)
    m = re.search(r"_sh([pm])(\d+)p(\d+)", name)
    if not m:
        return None
    sign = 1.0 if m.group(1) == "p" else -1.0
    return sign * float(f"{m.group(2)}.{m.group(3)}")


def eval_parse_episode(x):
    name = _ckpt_name(x)
    m = re.search(r"_ep(\d+)", name)
    return int(m.group(1)) if m else None


def eval_parse_step(x):
    name = _ckpt_name(x)
    m = re.search(r"_step(\d+)", name)
    return int(m.group(1)) if m else None


# -----------------------
# Discovery
# -----------------------
def eval_discover_actor_files(results_root: Path) -> pd.DataFrame:
    actors = sorted(results_root.rglob("*_actor.weights.h5"))
    rows = []

    for actor in actors:
        prefix = str(actor).replace("_actor.weights.h5", "")
        critic = Path(prefix + "_critic.weights.h5")
        if not critic.exists():
            continue

        parent_name = actor.parent.name
        if parent_name == "high_watermark_checkpoints":
            kind = "high_watermark"
        elif parent_name == "step_sharpe_checkpoints":
            kind = "step_sharpe"
        elif parent_name == "rare_models":
            kind = "rare"
        elif eval_parse_step(actor) is not None:
            kind = "periodic_step"
        else:
            kind = "root"

        rows.append({
            "actor_path": str(actor),
            "critic_path": str(critic),
            "checkpoint_prefix": prefix,
            "checkpoint_kind": kind,
            "episode": eval_parse_episode(actor),
            "step": eval_parse_step(actor),
            "sharpe_tag": eval_parse_sharpe_from_name(actor),
            "mtime": actor.stat().st_mtime,
        })

    cols = [
        "actor_path", "critic_path", "checkpoint_prefix",
        "checkpoint_kind", "episode", "step", "sharpe_tag", "mtime"
    ]
    return pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)


# -----------------------
# Basket selection
# -----------------------
def eval_select_ablation_basket(df_ckpt: pd.DataFrame) -> pd.DataFrame:
    picks = []

    if EVAL_INCLUDE_ROOT:
        root_df = df_ckpt[df_ckpt["checkpoint_kind"] == "root"].copy()
        if not root_df.empty:
            picks.append(root_df.sort_values("mtime", ascending=False).head(EVAL_TOP_ROOT))

    hw_df = df_ckpt[df_ckpt["checkpoint_kind"] == "high_watermark"].copy()
    if not hw_df.empty:
        hw_df["sharpe_rank_key"] = hw_df["sharpe_tag"].fillna(-np.inf)
        picks.append(
            hw_df.sort_values(
                ["sharpe_rank_key", "episode", "mtime"],
                ascending=[False, False, False]
            ).head(EVAL_TOP_HW)
        )

    periodic_df = df_ckpt[df_ckpt["checkpoint_kind"] == "periodic_step"].copy()
    if not periodic_df.empty:
        picks.append(
            periodic_df.sort_values(["step", "mtime"], ascending=[False, False]).head(EVAL_TOP_PERIODIC)
        )

    if EVAL_INCLUDE_RARE:
        rare_df = df_ckpt[df_ckpt["checkpoint_kind"] == "rare"].copy()
        if not rare_df.empty:
            rare_df["sharpe_rank_key"] = rare_df["sharpe_tag"].fillna(-np.inf)
            picks.append(
                rare_df.sort_values(
                    ["sharpe_rank_key", "episode", "mtime"],
                    ascending=[False, False, False]
                ).head(EVAL_TOP_RARE)
            )

    out = pd.concat(picks, ignore_index=True) if picks else pd.DataFrame(columns=df_ckpt.columns)

    # Force include specific episodes (prefer high_watermark/root)
    if EVAL_FORCE_EPISODES:
        forced = df_ckpt[
            (df_ckpt["episode"].isin(EVAL_FORCE_EPISODES)) &
            (df_ckpt["checkpoint_kind"].isin(["high_watermark", "root"]))
        ].copy()
        if not forced.empty:
            out = pd.concat([out, forced], ignore_index=True)

    out = out.drop_duplicates(subset=["checkpoint_prefix"]).reset_index(drop=True)
    out = out.sort_values(
        ["checkpoint_kind", "episode", "sharpe_tag", "step", "mtime"],
        ascending=[True, True, False, False, False]
    ).reset_index(drop=True)

    return out


# -----------------------
# Run
# -----------------------
eval_ckpt_df = eval_discover_actor_files(EVAL_RESULTS_ROOT)
if eval_ckpt_df.empty:
    raise RuntimeError(f"No valid actor+critic checkpoint pairs found under {EVAL_RESULTS_ROOT}")

print("All checkpoint pairs:", len(eval_ckpt_df))
print("By kind:", eval_ckpt_df["checkpoint_kind"].value_counts().to_dict())

eval_ablation_ckpts = eval_select_ablation_basket(eval_ckpt_df)

found_forced = sorted(
    set(eval_ablation_ckpts["episode"].dropna().astype(int).tolist()) & set(EVAL_FORCE_EPISODES)
)
missing_forced = sorted(set(EVAL_FORCE_EPISODES) - set(found_forced))

print("Selected for ablation:", len(eval_ablation_ckpts))
print("Forced requested:", EVAL_FORCE_EPISODES)
print("Forced found:", found_forced)
print("Forced missing:", missing_forced)

display(
    eval_ablation_ckpts[
        ["checkpoint_kind", "episode", "step", "sharpe_tag", "actor_path"]
    ].head(100)
)

All checkpoint pairs: 83
By kind: {'high_watermark': 83}
Selected for ablation: 15
Forced requested: [5, 25, 40, 45, 50, 154, 164]
Forced found: [5, 25, 40, 45, 50, 154, 164]
Forced missing: []


,checkpoint_kind,episode,step,sharpe_tag,actor_path
0,high_watermark,3,None,1.839,/content/eval_restore/tcn_fusion_results/high_...
1,high_watermark,5,None,1.331,/content/eval_restore/tcn_fusion_results/high_...
2,high_watermark,8,None,2.385,/content/eval_restore/tcn_fusion_results/high_...
3,high_watermark,9,None,1.889,/content/eval_restore/tcn_fusion_results/high_...
4,high_watermark,14,None,1.814,/content/eval_restore/tcn_fusion_results/high_...
5,high_watermark,21,None,3.142,/content/eval_restore/tcn_fusion_results/high_...
6,high_watermark,24,None,2.370,/content/eval_restore/tcn_fusion_results/high_...
7,high_watermark,25,None,1.342,/content/eval_restore/tcn_fusion_results/high_...
8,high_watermark,31,None,2.409,/content/eval_restore/tcn_fusion_results/high_...
9,high_watermark,40,None,1.350,/content/eval_restore/tcn_fusion_results/high_...


## 8) Evaluate ablation basket (deterministic + stochastic)

In [ ]:
from src.notebook_helpers.tcn_phase1 import (
    load_run_checkpoint_prefixes_from_metadata,
    preflight_checkpoint_loadability,
)

run_prefixes = load_run_checkpoint_prefixes_from_metadata(
    EVAL_METADATA_PATH,
    results_root=EVAL_RESULTS_ROOT,
    allowed_types={"high_watermark", "deterministic_validation_high_watermark", "final_high_watermark_style"},
    require_both_files=True,
)

# Fallback for older metadata
if not run_prefixes:
    print("ℹ️ No run-scoped checkpoint records in metadata; falling back to discovered checkpoints.")
    if "eval_ckpt_df" not in globals() or eval_ckpt_df is None or len(eval_ckpt_df) == 0:
        eval_ckpt_df = eval_discover_actor_files(EVAL_RESULTS_ROOT)

    fallback_df = eval_ckpt_df[eval_ckpt_df["checkpoint_kind"].isin(["high_watermark", "root"])].copy()
    run_prefixes = fallback_df["checkpoint_prefix"].dropna().unique().tolist()

print("run checkpoints:", len(run_prefixes))

preflight_df = preflight_checkpoint_loadability(
    checkpoint_prefixes=run_prefixes,
    phase1_data=eval_phase1_data,
    config=eval_config,
    random_seed=EVAL_RANDOM_SEED,
    use_covariance=True,
    architecture=eval_config["agent_params"]["actor_critic_type"],
)

if preflight_df is None or preflight_df.empty:
    print("⚠️ preflight returned empty; keeping current eval_ablation_ckpts")
else:
    display(preflight_df.head())
    if "compatible" in preflight_df.columns and "checkpoint_prefix" in preflight_df.columns:
        compatible_prefixes = set(preflight_df.loc[preflight_df["compatible"], "checkpoint_prefix"])
        eval_ablation_ckpts = eval_ablation_ckpts[
            eval_ablation_ckpts["checkpoint_prefix"].isin(compatible_prefixes)
        ].reset_index(drop=True)
        print("compatible selected:", len(eval_ablation_ckpts))

In [ ]:
display(
    eval_ablation_ckpts[["checkpoint_kind", "episode", "step", "sharpe_tag", "checkpoint_prefix"]]
    .sort_values(["episode", "step"], na_position="last")
    .reset_index(drop=True)
)

In [14]:
# before running evaluate loop
eval_config["training_params"]["num_parallel_envs"] = 1
eval_config["training_params"]["evaluation_action_execution_beta"] = 0.25
eval_config["training_params"]["evaluation_turnover_penalty_scalar"] = 0.2
eval_config["environment_params"]["action_execution_beta"] = 0.25
print("eval num_parallel_envs:", eval_config["training_params"]["num_parallel_envs"])
print("eval action_execution_beta:", eval_config["training_params"]["evaluation_action_execution_beta"])
print("eval turnover_penalty_scalar:", eval_config["training_params"]["evaluation_turnover_penalty_scalar"])

print("eval memory/regime/distributional:", {
    "recurrent_memory_enabled": eval_config["agent_params"].get("recurrent_memory_enabled"),
    "regime_conditioning_enabled": eval_config["agent_params"].get("regime_conditioning_enabled"),
    "state_augmentation_enabled": eval_config["agent_params"].get("state_augmentation_enabled"),
    "distributional_critic_enabled": eval_config["agent_params"].get("distributional_critic_enabled"),
    "distributional_num_quantiles": eval_config["agent_params"].get("distributional_num_quantiles"),
})
print("eval popart/reward-decomp:", {
    "popart_enabled": eval_config["agent_params"].get("ppo_params", {}).get("popart_enabled"),
    "multi_horizon_reward_enabled": eval_config["agent_params"].get("ppo_params", {}).get("multi_horizon_reward_enabled"),
    "multi_horizon_reward_coef": eval_config["agent_params"].get("ppo_params", {}).get("multi_horizon_reward_coef"),
})


eval num_parallel_envs: 1
eval action_execution_beta: 0.15
eval turnover_penalty_scalar: 0.2


In [15]:
def eval_run_one_checkpoint(
    eval_cfg,
    phase1_data,
    ckpt_prefix,
    seed=42,
    num_eval_runs_override=None,
    stochastic_episode_length_limit_override=None,
    save_logs_override=None,
    save_artifacts_override=None,
):
    stub = create_experiment6_result_stub(
        random_seed=seed,
        use_covariance=True,
        architecture=eval_cfg["agent_params"]["actor_critic_type"],
        checkpoint_path=ckpt_prefix,
        agent_config=copy.deepcopy(eval_cfg["agent_params"]),
        base_agent_params=None,
    )

    num_eval_runs = EVAL_NUM_STOCHASTIC_RUNS if num_eval_runs_override is None else int(num_eval_runs_override)
    stochastic_episode_limit = (
        EVAL_STOCHASTIC_EPISODE_LIMIT
        if stochastic_episode_length_limit_override is None
        else int(stochastic_episode_length_limit_override)
    )
    save_logs = EVAL_SAVE_LOGS if save_logs_override is None else bool(save_logs_override)
    save_artifacts = EVAL_SAVE_ARTIFACTS if save_artifacts_override is None else bool(save_artifacts_override)

    return evaluate_experiment6_checkpoint(
        experiment6=stub,
        phase1_data=phase1_data,
        config=eval_cfg,
        random_seed=seed,
        checkpoint_path_override=ckpt_prefix,
        deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
        num_eval_runs=num_eval_runs,
        stochastic_eval_mode=EVAL_STOCHASTIC_MODE,
        stochastic_episode_length_limit=stochastic_episode_limit,
        save_eval_logs=save_logs,
        save_eval_artifacts=save_artifacts,
    )


In [18]:
# -----------------------------------------------------------------------------
# Evaluation subset controls + two-stage offline selection controls
# -----------------------------------------------------------------------------
# Checkpoint subset modes:
# - "all": run full eval_ablation_ckpts
# - "index_range": run by row index range
# - "episode_range": checkpoints with episode in [min_ep, max_ep]
# - "episode_list": only specific episodes
EVAL_RUN_MODE = "all"
EVAL_INDEX_RANGE = (0, 9999)
EVAL_EPISODE_RANGE = (1, 9999)
EVAL_EPISODE_LIST = [5, 25, 40, 45, 50, 154, 164]

# Optional: clear previous results each run
EVAL_RESET_RESULTS = True

# Two-stage selection flow (fast train, heavy offline eval)
EVAL_TWO_STAGE_OFFLINE_SELECTION = True

# Stage 1: deterministic sweep grid
EVAL_SWEEP_HORIZONS = [252, 504, 756, 1008]
EVAL_SWEEP_START_OFFSETS = [0, 63, 126, 252]  # trading-day offsets into test period
EVAL_SWEEP_DD_PENALTY = 0.25
EVAL_SWEEP_SHARPE_WEIGHT = 1.00
EVAL_SWEEP_RETURN_WEIGHT = 0.60     # total-return contribution in selection score
EVAL_SWEEP_MDD_WEIGHT = 0.25        # drawdown penalty weight
EVAL_SWEEP_SHARPE_STD_WEIGHT = 0.10 # stability penalty across start dates
EVAL_DETERMINISTIC_SWEEP_SAVE_LOGS = False
EVAL_DETERMINISTIC_SWEEP_SAVE_ARTIFACTS = False

# Stage 2: stochastic reranking on horizon winners
EVAL_WINNER_STOCHASTIC_RUNS = 8
EVAL_WINNER_STOCHASTIC_EPISODE_LIMIT = 252
EVAL_WINNER_START_OFFSETS = [0, 63]  # evaluate winners on these starts
EVAL_HORIZON_AGG_WEIGHTS = {252: 0.15, 504: 0.25, 756: 0.30, 1008: 0.30}

# -----------------------------------------------------------------------------
# Build selected checkpoint frame
# -----------------------------------------------------------------------------
if EVAL_RUN_MODE == "all":
    eval_run_ckpts = eval_ablation_ckpts.copy()
elif EVAL_RUN_MODE == "index_range":
    i0, i1 = EVAL_INDEX_RANGE
    eval_run_ckpts = eval_ablation_ckpts.iloc[i0:i1 + 1].copy()
elif EVAL_RUN_MODE == "episode_range":
    ep0, ep1 = EVAL_EPISODE_RANGE
    tmp = eval_ablation_ckpts.copy()
    tmp["_ep_int"] = pd.to_numeric(tmp["episode"], errors="coerce")
    eval_run_ckpts = tmp[(tmp["_ep_int"] >= ep0) & (tmp["_ep_int"] <= ep1)].drop(columns=["_ep_int"]).copy()
elif EVAL_RUN_MODE == "episode_list":
    want = set(int(x) for x in EVAL_EPISODE_LIST)
    tmp = eval_ablation_ckpts.copy()
    tmp["_ep_int"] = pd.to_numeric(tmp["episode"], errors="coerce").astype("Int64")
    eval_run_ckpts = tmp[tmp["_ep_int"].isin(want)].drop(columns=["_ep_int"]).copy()
else:
    raise ValueError(f"Unknown EVAL_RUN_MODE: {EVAL_RUN_MODE}")

eval_run_ckpts = eval_run_ckpts.reset_index(drop=True)

print(f"Mode: {EVAL_RUN_MODE}")
print(f"Two-stage offline selection: {EVAL_TWO_STAGE_OFFLINE_SELECTION}")
print(f"Selected checkpoints: {len(eval_run_ckpts)} / {len(eval_ablation_ckpts)}")
print("Sweep objective weights:", {
    "w_sharpe": EVAL_SWEEP_SHARPE_WEIGHT,
    "w_return": EVAL_SWEEP_RETURN_WEIGHT,
    "w_mdd": EVAL_SWEEP_MDD_WEIGHT,
    "w_sharpe_std": EVAL_SWEEP_SHARPE_STD_WEIGHT,
})
display(eval_run_ckpts[["checkpoint_kind", "episode", "step", "sharpe_tag"]].head(20))


Mode: episode_list
Selected checkpoints: 7 / 15


,checkpoint_kind,episode,step,sharpe_tag
0,high_watermark,5,None,1.331
1,high_watermark,25,None,1.342
2,high_watermark,40,None,1.350
3,high_watermark,45,None,1.357
4,high_watermark,50,None,1.372
5,high_watermark,154,None,1.602
6,high_watermark,164,None,0.812



[1/7] Evaluating: high_watermark__ep0005__step-00001

LOADING CUSTOM CHECKPOINT: /content/eval_restore/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00005_shp1p331
✅ Found actor weights: /content/eval_restore/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00005_shp1p331_actor.weights.h5
✅ Found critic weights: /content/eval_restore/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00005_shp1p331_critic.weights.h5
🏗️ Recreating evaluation environments...
   🧭 Checkpoint architecture: TCN_FUSION (attention=True, fusion=True, source=path)
   🧱 Eval TCN stack: filters=[64, 96, 128, 128, 128] | kernel=5 | dilations=[1, 2, 4, 8, 16] | dropout=0.15
   🧩 Eval fusion core: embed=128 | heads=4 | dropout=0.1
   🔀 Eval mixer (A4): enabled=True | layers=2 | expansion=2.0 | dropout=0.1
   🎯 Eval alpha head (A3): dims=[128, 64] | dropout=0.05
   🎛️ Eval dirichlet: activation=softplus | temperature=1.0 | adaptive_temp=True | adaptive_base=0.9 | adaptive_slope=0

In [16]:
# -----------------------------------------------------------------------------
# Run evaluation
# -----------------------------------------------------------------------------
if EVAL_RESET_RESULTS or "eval_evaluations" not in globals():
    eval_evaluations = {}
if EVAL_RESET_RESULTS or "eval_failures" not in globals():
    eval_failures = {}


def _eval_checkpoint_label(row: pd.Series) -> str:
    ep = int(row["episode"]) if pd.notna(row.get("episode")) else -1
    st = int(row["step"]) if pd.notna(row.get("step")) else -1
    return f"{row['checkpoint_kind']}__ep{ep:04d}__step{st:06d}"


def _eval_make_phase1_slice(base_phase1, start_offset: int, horizon_days: int):
    test_df = base_phase1.test_df.copy()
    test_df["Date"] = pd.to_datetime(test_df["Date"])
    unique_dates = pd.Series(test_df["Date"].dropna().unique()).sort_values().reset_index(drop=True)

    if start_offset >= len(unique_dates):
        return None, None

    end_idx = min(len(unique_dates), int(start_offset) + int(horizon_days))
    win_dates = unique_dates.iloc[int(start_offset):end_idx]
    if len(win_dates) < max(60, int(horizon_days * 0.5)):
        return None, None

    d0 = pd.to_datetime(win_dates.iloc[0])
    d1 = pd.to_datetime(win_dates.iloc[-1])

    sliced_df = test_df[(test_df["Date"] >= d0) & (test_df["Date"] <= d1)].copy()
    if sliced_df.empty:
        return None, None

    phase1_slice = copy.deepcopy(base_phase1)
    phase1_slice.test_df = sliced_df
    phase1_slice.test_start_date = d0

    meta = {
        "start_date": str(d0.date()),
        "end_date": str(d1.date()),
        "n_days": int(len(win_dates)),
    }
    return phase1_slice, meta


if not EVAL_TWO_STAGE_OFFLINE_SELECTION:
    for i, row in eval_run_ckpts.iterrows():
        label = _eval_checkpoint_label(row)
        prefix = row["checkpoint_prefix"]

        print(f"\n[{i+1}/{len(eval_run_ckpts)}] Evaluating: {label}")
        try:
            ev = eval_run_one_checkpoint(eval_config, eval_phase1_data, prefix, seed=EVAL_RANDOM_SEED)
            eval_evaluations[label] = ev
        except Exception as e:
            eval_failures[label] = f"{type(e).__name__}: {e}"
            print(f"❌ Failed {label}: {eval_failures[label]}")
else:
    print("\n🔎 Stage 1/2: deterministic sweep (no stochastic during sweep)")
    det_records = []

    for i, row in eval_run_ckpts.iterrows():
        ckpt_label = _eval_checkpoint_label(row)
        prefix = row["checkpoint_prefix"]
        print(f"\n[{i+1}/{len(eval_run_ckpts)}] Sweep checkpoint: {ckpt_label}")

        for horizon in EVAL_SWEEP_HORIZONS:
            for start_offset in EVAL_SWEEP_START_OFFSETS:
                phase1_slice, meta = _eval_make_phase1_slice(eval_phase1_data, start_offset, horizon)
                if phase1_slice is None:
                    continue

                seed = int(EVAL_RANDOM_SEED + horizon * 1000 + start_offset)
                run_key = f"{ckpt_label}__h{horizon}__s{start_offset:03d}"
                try:
                    ev = eval_run_one_checkpoint(
                        eval_config,
                        phase1_slice,
                        prefix,
                        seed=seed,
                        num_eval_runs_override=0,
                        stochastic_episode_length_limit_override=min(EVAL_STOCHASTIC_EPISODE_LIMIT, horizon),
                        save_logs_override=EVAL_DETERMINISTIC_SWEEP_SAVE_LOGS,
                        save_artifacts_override=EVAL_DETERMINISTIC_SWEEP_SAVE_ARTIFACTS,
                    )
                    dm = ev.deterministic_metrics or {}
                    det_records.append(
                        {
                            "checkpoint_label": ckpt_label,
                            "checkpoint_prefix": prefix,
                            "horizon_days": int(horizon),
                            "start_offset": int(start_offset),
                            "start_date": meta["start_date"],
                            "end_date": meta["end_date"],
                            "n_days": meta["n_days"],
                            "det_sharpe": float(dm.get("sharpe_ratio", np.nan)),
                            "det_return": float(dm.get("total_return", np.nan)),
                            "det_mdd": float(dm.get("max_drawdown_abs", np.nan)),
                            "det_turnover": float(dm.get("turnover", np.nan)),
                        }
                    )
                except Exception as e:
                    eval_failures[run_key] = f"{type(e).__name__}: {e}"

    if not det_records:
        raise RuntimeError("No deterministic sweep records were produced.")

    eval_det_sweep_df = pd.DataFrame(det_records)
    display(eval_det_sweep_df.head(20))

    eval_det_agg_df = (
        eval_det_sweep_df
        .groupby(["checkpoint_label", "checkpoint_prefix", "horizon_days"], as_index=False)
        .agg(
            starts_used=("start_offset", "nunique"),
            det_sharpe_median=("det_sharpe", "median"),
            det_sharpe_mean=("det_sharpe", "mean"),
            det_sharpe_std=("det_sharpe", "std"),
            det_return_median=("det_return", "median"),
            det_mdd_median=("det_mdd", "median"),
            det_turnover_median=("det_turnover", "median"),
        )
    )
    eval_det_agg_df["selection_score"] = (
        EVAL_SWEEP_SHARPE_WEIGHT * eval_det_agg_df["det_sharpe_median"].fillna(-999)
        + EVAL_SWEEP_RETURN_WEIGHT * eval_det_agg_df["det_return_median"].fillna(-1.0)
        - EVAL_SWEEP_MDD_WEIGHT * eval_det_agg_df["det_mdd_median"].fillna(1.0)
        - EVAL_SWEEP_SHARPE_STD_WEIGHT * eval_det_agg_df["det_sharpe_std"].fillna(0.0)
    )

    eval_horizon_winners_df = (
        eval_det_agg_df
        .sort_values(["horizon_days", "selection_score", "det_sharpe_median"], ascending=[True, False, False])
        .groupby("horizon_days", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    print("\n🏅 Deterministic sweep winners by horizon")
    display(eval_horizon_winners_df)

    eval_det_agg_df["horizon_weight"] = eval_det_agg_df["horizon_days"].map(EVAL_HORIZON_AGG_WEIGHTS).fillna(0.0)
    overall_df = (
        eval_det_agg_df
        .groupby(["checkpoint_label", "checkpoint_prefix"], as_index=False)
        .apply(lambda g: pd.Series({
            "overall_weighted_score": float((g["selection_score"] * g["horizon_weight"]).sum()),
            "overall_weighted_sharpe": float((g["det_sharpe_median"] * g["horizon_weight"]).sum()),
            "overall_weighted_return": float((g["det_return_median"] * g["horizon_weight"]).sum()),
            "overall_weighted_mdd": float((g["det_mdd_median"] * g["horizon_weight"]).sum()),
            "horizons_covered": int(g["horizon_days"].nunique()),
        }))
        .reset_index(drop=True)
        .sort_values("overall_weighted_score", ascending=False)
    )
    eval_overall_winner_df = overall_df.head(1).copy()
    print("\n🏆 Overall robust winner across horizons/start dates")
    display(eval_overall_winner_df)

    print("\n🎲 Stage 2/2: stochastic reranking on horizon winners")
    winner_records = []

    finalists_df = eval_horizon_winners_df[["checkpoint_label", "checkpoint_prefix", "horizon_days"]].copy()
    if "eval_overall_winner_df" in globals() and not eval_overall_winner_df.empty:
        ow = eval_overall_winner_df.iloc[0]
        for hz in EVAL_SWEEP_HORIZONS:
            finalists_df = pd.concat([
                finalists_df,
                pd.DataFrame([{
                    "checkpoint_label": ow["checkpoint_label"],
                    "checkpoint_prefix": ow["checkpoint_prefix"],
                    "horizon_days": int(hz),
                }])
            ], ignore_index=True)
    finalists_df = finalists_df.drop_duplicates(subset=["checkpoint_label", "checkpoint_prefix", "horizon_days"]).reset_index(drop=True)

    for _, winner in finalists_df.iterrows():
        horizon = int(winner["horizon_days"])
        prefix = winner["checkpoint_prefix"]
        ckpt_label = winner["checkpoint_label"]

        for start_offset in EVAL_WINNER_START_OFFSETS:
            phase1_slice, meta = _eval_make_phase1_slice(eval_phase1_data, start_offset, horizon)
            if phase1_slice is None:
                continue

            label = f"h{horizon}__s{int(start_offset):03d}__{ckpt_label}"
            try:
                ev = eval_run_one_checkpoint(
                    eval_config,
                    phase1_slice,
                    prefix,
                    seed=int(EVAL_RANDOM_SEED + 500_000 + horizon * 1000 + start_offset),
                    num_eval_runs_override=EVAL_WINNER_STOCHASTIC_RUNS,
                    stochastic_episode_length_limit_override=min(EVAL_WINNER_STOCHASTIC_EPISODE_LIMIT, horizon),
                    save_logs_override=EVAL_SAVE_LOGS,
                    save_artifacts_override=EVAL_SAVE_ARTIFACTS,
                )
                eval_evaluations[label] = ev

                dm = ev.deterministic_metrics or {}
                sto = ev.stochastic_results if isinstance(ev.stochastic_results, pd.DataFrame) else pd.DataFrame()
                winner_records.append(
                    {
                        "label": label,
                        "horizon_days": horizon,
                        "start_offset": int(start_offset),
                        "start_date": meta["start_date"],
                        "end_date": meta["end_date"],
                        "checkpoint_prefix": prefix,
                        "det_sharpe": float(dm.get("sharpe_ratio", np.nan)),
                        "det_return": float(dm.get("total_return", np.nan)),
                        "det_mdd": float(dm.get("max_drawdown_abs", np.nan)),
                        "sto_sharpe_mean": float(sto["sharpe_ratio"].mean()) if (not sto.empty and "sharpe_ratio" in sto.columns) else np.nan,
                        "sto_sharpe_std": float(sto["sharpe_ratio"].std()) if (not sto.empty and "sharpe_ratio" in sto.columns) else np.nan,
                    }
                )
            except Exception as e:
                eval_failures[label] = f"{type(e).__name__}: {e}"
                print(f"❌ Failed winner eval {label}: {eval_failures[label]}")

    eval_winner_table = pd.DataFrame(winner_records)
    if not eval_winner_table.empty:
        print("\n✅ Winner checkpoint evaluations (with stochastic runs)")
        display(eval_winner_table.sort_values(["horizon_days", "det_sharpe"], ascending=[True, False]))

print("✅ Completed evaluations:", len(eval_evaluations))
print("⚠️ Failed evaluations:", len(eval_failures))

if eval_failures:
    print("\nFailure samples:")
    for k, v in list(eval_failures.items())[:10]:
        print(" -", k, "->", v)


Mode: episode_list
Selected checkpoints: 1 / 15


,checkpoint_kind,episode,step,sharpe_tag
0,high_watermark,25,None,1.342



[1/1] Evaluating: high_watermark__ep0025__step-00001

LOADING CUSTOM CHECKPOINT: /content/eval_restore/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00025_shp1p342
✅ Found actor weights: /content/eval_restore/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00025_shp1p342_actor.weights.h5
✅ Found critic weights: /content/eval_restore/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00025_shp1p342_critic.weights.h5
🏗️ Recreating evaluation environments...
   🧭 Checkpoint architecture: TCN_FUSION (attention=True, fusion=True, source=path)
   🧱 Eval TCN stack: filters=[64, 96, 128, 128, 128] | kernel=5 | dilations=[1, 2, 4, 8, 16] | dropout=0.15
   🧩 Eval fusion core: embed=128 | heads=4 | dropout=0.1
   🔀 Eval mixer (A4): enabled=True | layers=2 | expansion=2.0 | dropout=0.1
   🎯 Eval alpha head (A3): dims=[128, 64] | dropout=0.05
   🎛️ Eval dirichlet: activation=softplus | temperature=1.0 | adaptive_temp=True | adaptive_base=0.9 | adaptive_slope=0

## 9) Ablation table and leaderboard

In [19]:
if not eval_evaluations:
    raise RuntimeError('No successful evaluations to summarize.')

eval_ablation_table = build_ablation_table(eval_evaluations)

display(eval_ablation_table.head(30))

# Deterministic-first leaderboard view
eval_leaderboard = eval_ablation_table.copy()
eval_leaderboard['risk_adjusted_score'] = (
    eval_leaderboard['det_sharpe'].fillna(-999)
    - 0.5 * eval_leaderboard['det_max_drawdown'].fillna(1.0)
    - 0.1 * eval_leaderboard['det_turnover'].fillna(1.0)
)
eval_leaderboard = eval_leaderboard.sort_values(['risk_adjusted_score', 'det_sharpe'], ascending=False).reset_index(drop=True)

print('Top by risk-adjusted score:')
display(eval_leaderboard.head(10))

,label,checkpoint_description,det_sharpe,det_sortino,det_max_drawdown,det_volatility,det_turnover,sto_mean_sharpe,sto_std_sharpe,sto_mean_return,sto_mean_max_drawdown
1,high_watermark__ep0025__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.826842,1.196328,0.131794,0.128055,0.000826,0.837884,0.372402,0.246873,0.111713
5,high_watermark__ep0154__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.825866,1.193394,0.139469,0.132199,0.000633,0.880035,0.332599,0.269441,0.114033
0,high_watermark__ep0005__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.820566,1.189808,0.133354,0.129850,0.000823,0.947110,0.344340,0.281267,0.109351
6,high_watermark__ep0164__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.812095,1.173995,0.137550,0.131683,0.000718,0.892527,0.302721,0.273831,0.107485
2,high_watermark__ep0040__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.809373,1.170845,0.128883,0.127674,0.000788,0.839153,0.388403,0.248663,0.106930
4,high_watermark__ep0050__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.807702,1.166887,0.123597,0.124841,0.000776,0.822621,0.341829,0.239666,0.108681
3,high_watermark__ep0045__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.806867,1.166236,0.127322,0.126058,0.000814,0.855346,0.358057,0.251245,0.111430


Top by risk-adjusted score:


,label,checkpoint_description,det_sharpe,det_sortino,det_max_drawdown,det_volatility,det_turnover,sto_mean_sharpe,sto_std_sharpe,sto_mean_return,sto_mean_max_drawdown,risk_adjusted_score
0,high_watermark__ep0025__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.826842,1.196328,0.131794,0.128055,0.000826,0.837884,0.372402,0.246873,0.111713,0.760862
1,high_watermark__ep0154__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.825866,1.193394,0.139469,0.132199,0.000633,0.880035,0.332599,0.269441,0.114033,0.756068
2,high_watermark__ep0005__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.820566,1.189808,0.133354,0.129850,0.000823,0.947110,0.344340,0.281267,0.109351,0.753807
3,high_watermark__ep0050__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.807702,1.166887,0.123597,0.124841,0.000776,0.822621,0.341829,0.239666,0.108681,0.745826
4,high_watermark__ep0040__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.809373,1.170845,0.128883,0.127674,0.000788,0.839153,0.388403,0.248663,0.106930,0.744853
5,high_watermark__ep0164__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.812095,1.173995,0.137550,0.131683,0.000718,0.892527,0.302721,0.273831,0.107485,0.743248
6,high_watermark__ep0045__step-00001,Custom checkpoint (/content/eval_restore/tcn_f...,0.806867,1.166236,0.127322,0.126058,0.000814,0.855346,0.358057,0.251245,0.111430,0.743124


## 10) Build industry baseline returns (equal-weight and cash)

In [21]:
def eval_identify_asset_column(df: pd.DataFrame):
    candidates = ['Ticker', 'ticker', 'tic', 'asset', 'Asset', 'symbol', 'Symbol']
    for c in candidates:
        if c in df.columns:
            return c
    return None


def eval_identify_return_column(df: pd.DataFrame):
    candidates = ['LogReturn_1d', 'log_return_1d', 'Return_1d', 'return_1d', 'daily_return']
    for c in candidates:
        if c in df.columns:
            return c
    return None


def eval_fetch_sp500_returns(start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.Series:
    """
    Fetch S&P500 daily simple returns for benchmark comparison.
    Primary source: yfinance '^GSPC'.
    Fallback: empty series if fetch fails.
    """
    try:
        import yfinance as yf
    except Exception:
        try:
            !pip -q install yfinance
            import yfinance as yf
        except Exception:
            print('⚠️ Could not install/import yfinance; SP500 benchmark disabled.')
            return pd.Series(dtype=float)

    try:
        df = yf.download('^GSPC', start=str(start_date.date()), end=str((end_date + pd.Timedelta(days=1)).date()), auto_adjust=True, progress=False)
        if df is None or df.empty:
            print('⚠️ SP500 download returned empty data.')
            return pd.Series(dtype=float)
        try:
            close_obj = df['Close']
        except Exception:
            if isinstance(df.columns, pd.MultiIndex) and 'Close' in df.columns.get_level_values(0):
                close_obj = df.xs('Close', axis=1, level=0)
            else:
                print('⚠️ SP500 download missing Close column.')
                return pd.Series(dtype=float)
        if isinstance(close_obj, pd.DataFrame):
            # yfinance can return (n,1) DataFrame for a single ticker; squeeze to 1D.
            close = close_obj.iloc[:, 0] if close_obj.shape[1] == 1 else close_obj.mean(axis=1)
        else:
            close = close_obj
        close = pd.to_numeric(close, errors='coerce').dropna()
        ret = close.pct_change().dropna().astype(float)
        ret.index = pd.to_datetime(ret.index)
        return ret
    except Exception as e:
        print(f'⚠️ SP500 fetch failed: {type(e).__name__}: {e}')
        return pd.Series(dtype=float)


def eval_build_baselines_from_phase1(phase1_data):
    test_df = phase1_data.test_df.copy()
    if 'Date' not in test_df.columns:
        raise ValueError('test_df must contain Date column')

    ret_col = eval_identify_return_column(test_df)
    if ret_col is None:
        raise ValueError('Could not identify return column in test_df')

    if 'LogReturn' in ret_col or 'log' in ret_col.lower():
        test_df['_simple_ret'] = np.expm1(test_df[ret_col].astype(float))
    else:
        test_df['_simple_ret'] = test_df[ret_col].astype(float)

    # Industry baseline 1: equal-weight over available assets each day
    eqw = (
        test_df.groupby('Date')['_simple_ret']
        .mean()
        .sort_index()
        .astype(float)
    )

    # Industry baseline 2: cash (0% daily return)
    cash = pd.Series(np.zeros(len(eqw)), index=pd.to_datetime(eqw.index), name='cash')

    # Industry baseline 3: S&P 500 (^GSPC)
    dt_index = pd.to_datetime(eqw.index)
    sp500_ret = eval_fetch_sp500_returns(dt_index.min(), dt_index.max())
    if not sp500_ret.empty:
        # align to model dates; missing market holidays become 0 return for alignment stability
        sp500_ret = sp500_ret.reindex(dt_index).fillna(0.0)
    else:
        sp500_ret = pd.Series(dtype=float)

    # reset to plain 0..n index for compare_agent_vs_baseline
    eqw = eqw.reset_index(drop=True)
    cash = cash.reset_index(drop=True)
    sp500 = sp500_ret.reset_index(drop=True) if not sp500_ret.empty else pd.Series(dtype=float)
    return eqw, cash, sp500


eval_baseline_eqw, eval_baseline_cash, eval_baseline_sp500 = eval_build_baselines_from_phase1(eval_phase1_data)
print('Baseline lengths | EQW:', len(eval_baseline_eqw), 'Cash:', len(eval_baseline_cash), 'SP500:', len(eval_baseline_sp500))

Baseline lengths | EQW: 1103 Cash: 1103 SP500: 1103


## 11) Benchmark each evaluated checkpoint vs baselines

In [22]:
benchmark_rows = []
for label, ev in eval_evaluations.items():
    try:
        cmp_eqw = compare_agent_vs_baseline(ev, eval_baseline_eqw)
    except Exception as e:
        cmp_eqw = {'error': str(e)}

    try:
        cmp_cash = compare_agent_vs_baseline(ev, eval_baseline_cash)
    except Exception as e:
        cmp_cash = {'error': str(e)}

    try:
        if len(eval_baseline_sp500) > 0:
            cmp_sp500 = compare_agent_vs_baseline(ev, eval_baseline_sp500)
        else:
            cmp_sp500 = {'error': 'SP500 baseline unavailable'}
    except Exception as e:
        cmp_sp500 = {'error': str(e)}

    row = {
        'label': label,
        'det_sharpe': (ev.deterministic_metrics or {}).get('sharpe_ratio', np.nan),
        'det_return': (ev.deterministic_metrics or {}).get('annualized_return', np.nan),
        'det_mdd': (ev.deterministic_metrics or {}).get('max_drawdown_abs', np.nan),
        'det_turnover': (ev.deterministic_metrics or {}).get('turnover', np.nan),
    }

    for prefix, comp in [('eqw', cmp_eqw), ('cash', cmp_cash), ('sp500', cmp_sp500)]:
        if isinstance(comp, dict) and 'error' not in comp:
            for k, v in comp.items():
                row[f'{prefix}_{k}'] = v
        else:
            row[f'{prefix}_error'] = comp.get('error', 'unknown') if isinstance(comp, dict) else 'unknown'

    benchmark_rows.append(row)

eval_benchmark_df = pd.DataFrame(benchmark_rows)

display(eval_benchmark_df.sort_values('det_sharpe', ascending=False).head(20))

,label,det_sharpe,det_return,det_mdd,det_turnover,eqw_agent_sharpe,eqw_baseline_sharpe,eqw_agent_mean_return,eqw_baseline_mean_return,eqw_agent_volatility,...,cash_agent_mean_return,cash_baseline_mean_return,cash_agent_volatility,cash_baseline_volatility,sp500_agent_sharpe,sp500_baseline_sharpe,sp500_agent_mean_return,sp500_baseline_mean_return,sp500_agent_volatility,sp500_baseline_volatility
1,high_watermark__ep0025__step-00001,0.826842,0.124530,0.131794,0.000826,0.981489,5.999496,0.000499,0.912498,0.128055,...,0.000499,0.0,0.128055,0.0,0.981489,0.683834,0.000499,0.000472,0.128055,0.174051
5,high_watermark__ep0154__step-00001,0.825866,0.127634,0.139469,0.000633,0.975666,5.999496,0.000512,0.912498,0.132199,...,0.000512,0.0,0.132199,0.0,0.975666,0.683834,0.000512,0.000472,0.132199,0.174051
0,high_watermark__ep0005__step-00001,0.820566,0.125022,0.133354,0.000823,0.973076,5.999496,0.000501,0.912498,0.129850,...,0.000501,0.0,0.129850,0.0,0.973076,0.683834,0.000501,0.000472,0.129850,0.174051
6,high_watermark__ep0164__step-00001,0.812095,0.125190,0.137550,0.000718,0.962481,5.999496,0.000503,0.912498,0.131683,...,0.000503,0.0,0.131683,0.0,0.962481,0.683834,0.000503,0.000472,0.131683,0.174051
2,high_watermark__ep0040__step-00001,0.809373,0.121728,0.128883,0.000788,0.964482,5.999496,0.000489,0.912498,0.127674,...,0.000489,0.0,0.127674,0.0,0.964482,0.683834,0.000489,0.000472,0.127674,0.174051
4,high_watermark__ep0050__step-00001,0.807702,0.119328,0.123597,0.000776,0.966331,5.999496,0.000479,0.912498,0.124841,...,0.000479,0.0,0.124841,0.0,0.966331,0.683834,0.000479,0.000472,0.124841,0.174051
3,high_watermark__ep0045__step-00001,0.806867,0.120139,0.127322,0.000814,0.963964,5.999496,0.000482,0.912498,0.126058,...,0.000482,0.0,0.126058,0.0,0.963964,0.683834,0.000482,0.000472,0.126058,0.174051


## 12) Champion selection (production candidate)

In [23]:
if eval_benchmark_df.empty:
    raise RuntimeError('No benchmark rows available.')

# Balanced production-style objective: reward risk-adjusted return, penalize drawdown/turnover.
eval_benchmark_df['selection_score'] = (
    eval_benchmark_df['det_sharpe'].fillna(-999)
    + 0.2 * eval_benchmark_df['det_return'].fillna(0.0)
    - 0.7 * eval_benchmark_df['det_mdd'].fillna(1.0)
    - 0.1 * eval_benchmark_df['det_turnover'].fillna(1.0)
)

champion_row = eval_benchmark_df.sort_values('selection_score', ascending=False).iloc[0]
EVAL_CHAMPION_LABEL = champion_row['label']
EVAL_CHAMPION = eval_evaluations[EVAL_CHAMPION_LABEL]

print('🏆 Champion label:', EVAL_CHAMPION_LABEL)
print(champion_row[['det_sharpe', 'det_return', 'det_mdd', 'det_turnover', 'selection_score']])

🏆 Champion label: high_watermark__ep0025__step-00001
det_sharpe         0.826842
det_return          0.12453
det_mdd            0.131794
det_turnover       0.000826
selection_score    0.759409
Name: 1, dtype: object


## 13) Regime-sliced performance (champion vs equal-weight)

In [24]:
def eval_regime_tag(dates: pd.Series):
    d = pd.to_datetime(dates)
    conds = [
        (d <= pd.Timestamp('2020-02-19')),
        (d >= pd.Timestamp('2020-02-20')) & (d <= pd.Timestamp('2020-06-30')),
        (d >= pd.Timestamp('2020-07-01')) & (d <= pd.Timestamp('2021-12-31')),
        (d >= pd.Timestamp('2022-01-01')) & (d <= pd.Timestamp('2023-12-31')),
        (d >= pd.Timestamp('2024-01-01')),
    ]
    labels = ['pre_covid', 'covid_crash', 'post_covid_recovery', 'inflation_rates', 'recent']
    out = np.select(conds, labels, default='other')
    return pd.Series(out)


def eval_sharpe(x):
    x = pd.Series(x).dropna()
    if len(x) < 2:
        return np.nan
    std = x.std(ddof=1)
    if std <= 1e-12:
        return np.nan
    return np.sqrt(252.0) * x.mean() / std

# Build aligned daily return series for champion and baselines
champ_port = np.array(EVAL_CHAMPION.deterministic_portfolio)
champ_ret = pd.Series(np.diff(champ_port) / champ_port[:-1]).reset_index(drop=True)
eqw_ret = eval_baseline_eqw.reset_index(drop=True)
sp500_ret = eval_baseline_sp500.reset_index(drop=True) if len(eval_baseline_sp500) > 0 else pd.Series(dtype=float)

n_core = min(len(champ_ret), len(eqw_ret), len(eval_phase1_data.test_df['Date'].drop_duplicates()) - 1)
if len(sp500_ret) > 0:
    n = min(n_core, len(sp500_ret))
else:
    n = n_core

dates = pd.to_datetime(eval_phase1_data.test_df['Date'].drop_duplicates().sort_values()).reset_index(drop=True).iloc[1:n+1]

reg_df = pd.DataFrame({
    'Date': dates.reset_index(drop=True),
    'champion_ret': champ_ret.iloc[:n].reset_index(drop=True),
    'eqw_ret': eqw_ret.iloc[:n].reset_index(drop=True),
})
if len(sp500_ret) > 0:
    reg_df['sp500_ret'] = sp500_ret.iloc[:n].reset_index(drop=True)
else:
    reg_df['sp500_ret'] = np.nan

reg_df['regime'] = eval_regime_tag(reg_df['Date'])

regime_rows = []
for regime, g in reg_df.groupby('regime'):
    row = {
        'regime': regime,
        'n_days': len(g),
        'champion_sharpe': eval_sharpe(g['champion_ret']),
        'eqw_sharpe': eval_sharpe(g['eqw_ret']),
        'champion_total_return': float((1.0 + g['champion_ret']).prod() - 1.0),
        'eqw_total_return': float((1.0 + g['eqw_ret']).prod() - 1.0),
    }
    if g['sp500_ret'].notna().any():
        row['sp500_sharpe'] = eval_sharpe(g['sp500_ret'])
        row['sp500_total_return'] = float((1.0 + g['sp500_ret'].fillna(0.0)).prod() - 1.0)
    else:
        row['sp500_sharpe'] = np.nan
        row['sp500_total_return'] = np.nan
    regime_rows.append(row)

eval_regime_df = pd.DataFrame(regime_rows).sort_values('regime').reset_index(drop=True)
display(eval_regime_df)

,regime,n_days,champion_sharpe,eqw_sharpe,champion_total_return,eqw_total_return,sp500_sharpe,sp500_total_return
0,inflation_rates,501,0.583080,7.054031,0.160228,4.325694e+81,0.099748,0.000967
1,post_covid_recovery,184,2.024056,9.377625,0.162520,3.716401e+19,1.711066,0.157641
2,recent,417,1.205321,4.963357,0.239240,6.269167e+64,1.191766,0.359269


## 14) Statistical confidence: bootstrap Sharpe difference (champion - equal-weight)

In [25]:
def eval_block_bootstrap_sharpe_diff(agent_ret, base_ret, n_boot=2000, block=20, seed=42):
    rng = np.random.default_rng(seed)
    a = np.asarray(agent_ret, dtype=float)
    b = np.asarray(base_ret, dtype=float)
    n = min(len(a), len(b))
    a = a[:n]
    b = b[:n]

    def _sharpe(x):
        x = pd.Series(x).dropna()
        if len(x) < 2:
            return np.nan
        s = x.std(ddof=1)
        if s <= 1e-12:
            return np.nan
        return np.sqrt(252.0) * x.mean() / s

    diffs = []
    n_blocks = int(np.ceil(n / block))
    max_start = max(1, n - block + 1)

    for _ in range(n_boot):
        idx = []
        for __ in range(n_blocks):
            st = int(rng.integers(0, max_start))
            idx.extend(range(st, min(st + block, n)))
        idx = np.asarray(idx[:n])
        d = _sharpe(a[idx]) - _sharpe(b[idx])
        if np.isfinite(d):
            diffs.append(float(d))

    if not diffs:
        return {'n_boot_eff': 0, 'mean': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'p_le_zero': np.nan}

    diffs = np.asarray(diffs)
    return {
        'n_boot_eff': int(len(diffs)),
        'mean': float(np.mean(diffs)),
        'ci_low': float(np.quantile(diffs, 0.025)),
        'ci_high': float(np.quantile(diffs, 0.975)),
        'p_le_zero': float(np.mean(diffs <= 0.0)),
    }

bootstrap_eqw = eval_block_bootstrap_sharpe_diff(
    reg_df['champion_ret'].values,
    reg_df['eqw_ret'].values,
    n_boot=2000,
    block=20,
    seed=EVAL_RANDOM_SEED,
)

if reg_df['sp500_ret'].notna().any():
    bootstrap_sp500 = eval_block_bootstrap_sharpe_diff(
        reg_df['champion_ret'].values,
        reg_df['sp500_ret'].fillna(0.0).values,
        n_boot=2000,
        block=20,
        seed=EVAL_RANDOM_SEED,
    )
else:
    bootstrap_sp500 = {'n_boot_eff': 0, 'mean': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'p_le_zero': np.nan}

print('Bootstrap Sharpe diff (Champion - EQW):')
print(bootstrap_eqw)
print('Bootstrap Sharpe diff (Champion - SP500):')
print(bootstrap_sp500)

Bootstrap Sharpe diff (Champion - EQW):
{'n_boot_eff': 2000, 'mean': -5.224736139808108, 'ci_low': -6.6244366065758005, 'ci_high': -4.015129075907271, 'p_le_zero': 1.0}
Bootstrap Sharpe diff (Champion - SP500):
{'n_boot_eff': 2000, 'mean': 0.26305312886005777, 'ci_low': -0.32494760734641887, 'ci_high': 0.8174387412245373, 'p_le_zero': 0.184}


## 15) Training-diagnostics quality checks from CSV metrics
Uses saved CSVs to report KL stability, turnover drivers, and execution quality.

In [ ]:
diag_report = {}

if eval_latest_episodes_csv and Path(eval_latest_episodes_csv).exists():
    ep = pd.read_csv(eval_latest_episodes_csv)
    diag_report['episodes_rows'] = len(ep)

    if 'approx_kl' in ep.columns:
        kl = pd.to_numeric(ep['approx_kl'], errors='coerce').dropna()
        if len(kl):
            diag_report['approx_kl_mean'] = float(kl.mean())
            diag_report['approx_kl_p50'] = float(kl.quantile(0.50))
            diag_report['approx_kl_p90'] = float(kl.quantile(0.90))

    if {'episode_turnover_pct', 'approx_kl'}.issubset(ep.columns):
        x = pd.to_numeric(ep['episode_turnover_pct'], errors='coerce')
        y = pd.to_numeric(ep['approx_kl'], errors='coerce')
        valid = x.notna() & y.notna()
        if valid.any():
            diag_report['corr_turnoverpct_kl'] = float(np.corrcoef(x[valid], y[valid])[0, 1])

if eval_latest_step_diag_csv and Path(eval_latest_step_diag_csv).exists():
    sd = pd.read_csv(eval_latest_step_diag_csv)
    diag_report['step_diag_rows'] = len(sd)

    for col in ['l1_w_delta', 'turnover_penalty_contrib', 'tx_cost_contrib_reward_pts', 'action_realization_l1']:
        if col in sd.columns:
            s = pd.to_numeric(sd[col], errors='coerce').dropna()
            if len(s):
                diag_report[f'{col}_mean'] = float(s.mean())
                diag_report[f'{col}_p90'] = float(s.quantile(0.90))

print(json.dumps(diag_report, indent=2))

## 16) Save final evaluation package
Exports leaderboard, benchmark table, regime table, diagnostics, and champion metadata.

In [ ]:
eval_out_dir = EVAL_RESULTS_ROOT / 'logs'
eval_out_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')

eval_ablation_path = eval_out_dir / f'final_eval_ablation_{ts}.csv'
eval_benchmark_path = eval_out_dir / f'final_eval_benchmark_{ts}.csv'
eval_regime_path = eval_out_dir / f'final_eval_regime_{ts}.csv'
eval_diag_path = eval_out_dir / f'final_eval_diagnostics_{ts}.json'
eval_meta_path = eval_out_dir / f'final_eval_champion_{ts}.json'

eval_ablation_table.to_csv(eval_ablation_path, index=False)
eval_benchmark_df.to_csv(eval_benchmark_path, index=False)
eval_regime_df.to_csv(eval_regime_path, index=False)

with open(eval_diag_path, 'w', encoding='utf-8') as f:
    json.dump({
        'bootstrap_sharpe_diff_eqw': bootstrap_eqw,
        'bootstrap_sharpe_diff_sp500': bootstrap_sp500,
        'diagnostics': diag_report,
        'metadata_path': str(EVAL_METADATA_PATH),
    }, f, indent=2)

with open(eval_meta_path, 'w', encoding='utf-8') as f:
    json.dump({
        'champion_label': EVAL_CHAMPION_LABEL,
        'selection_row': champion_row.to_dict(),
        'deterministic_metrics': EVAL_CHAMPION.deterministic_metrics,
        'checkpoint_description': EVAL_CHAMPION.checkpoint_description,
    }, f, indent=2, default=str)

print('✅ Saved evaluation package:')
print('-', eval_ablation_path)
print('-', eval_benchmark_path)
print('-', eval_regime_path)
print('-', eval_diag_path)
print('-', eval_meta_path)